In [ ]:
import duckdb

# Simulate Snowflake Dynamic Table behavior using DuckDB
con = duckdb.connect(database=':memory:')

# Create Mock Silver Table
con.execute("""
CREATE TABLE omop_person (
    person_id BIGINT,
    gender_concept_id INT,
    person_source_value VARCHAR,
    updated_at TIMESTAMP
)
""")

# Insert initial data
con.execute("""
INSERT INTO omop_person VALUES 
(1, 8507, 'A', current_timestamp()),
(2, 8532, 'B', current_timestamp()),
(3, 8507, 'C', current_timestamp())
""")

print("SILVER TABLE (Base Data):")
print(con.execute("SELECT * FROM omop_person").df())

# Simulate Dynamic Table View (Gold Layer)
con.execute("""
CREATE VIEW gold_metrics AS 
SELECT 
    gender_concept_id,
    COUNT(DISTINCT person_id) as patient_count
FROM omop_person
GROUP BY gender_concept_id
""")

print("\nGOLD DYNAMIC TABLE (Lag 0 view):")
print(con.execute("SELECT * FROM gold_metrics").df())

# Simulate incremental arrival
con.execute("INSERT INTO omop_person VALUES (4, 8532, 'D', current_timestamp())")

print("\nGOLD DYNAMIC TABLE (After new record arrives in Silver):")
print(con.execute("SELECT * FROM gold_metrics").df())
